### Import Libraries


In [1]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait as WOW
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


In [2]:
from selenium.webdriver.support.ui import WebDriverWait as WDW #used this code to import wait time feauture and shortened as WDW 

### Setup and Configure Selenium WebDriver

In [3]:
print("Setting up Webddriver....")
chrome_opt = Options() # Intialize the chrome webdriver
chrome_opt.add_argument("--headless")
chrome_opt.add_argument("--disable-gpu")
chrome_opt.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.6778.265 Safari/537.36")
print("configuration done!")

Setting up Webddriver....
configuration done!


In [4]:
# Setting Up Webdriver: Installation and Initialization
print("Installing Chrome Webdriver")
service = Service(ChromeDriverManager().install())
print("Final Setup")
driver = webdriver.Chrome(service=service, options=chrome_opt)
print("done!!")


Installing Chrome Webdriver
Final Setup
done!!


### Making a connection to the webpage


In [5]:
URL = "https://www.framesdirect.com/eyeglasses/"

In [ ]:
print(f"Visiting {URL} page")
driver.get(URL)

# Further Instructions
try:
    print("Waiting for product tiles to load")
    WDW(driver, 20).until(EC.presence_of_element_located((By.CLASS_NAME, 'fd-cat'))) #Here we are using a class name from the body tag to ensure we are targeting the glasses lists.
    print("Done, Proceed!")
except TimeoutError as e:
    print(f"Expected tag did not load on time: {e}")

Visiting https://www.framesdirect.com/eyeglasses/ page
Waiting for product tiles to load
Done, Proceed!


In [31]:
content = driver.page_source
page = BeautifulSoup(content, "html.parser")

### Data Extraction

In [32]:
product_tiles = page.find_all("div", class_="prod-holder")
print(f"Found {len(product_tiles)} products")

Found 25 products


In [39]:
products = []

for tile in product_tiles:
    product_info = tile.find('div', class_='prod-image-holder')
#prod-image-holder
    if product_info:
        name_tag = product_info.find('div', class_='product_name')
        name = name_tag.text if name_tag else "Unknown"

        # brand
        brand_tag = product_info.find('div', class_='catalog-name')
        brand = brand_tag.text if brand_tag else "Unknown"
        
        # price
        price_container = product_info.find('div', class_='prod-bot')
        if price_container:
            # former price
            former_price_tag = price_container.find('div', class_='prod-catalog-retail-price')
            former_price = former_price_tag.text if former_price_tag else 'Unknown'
            # Current price
            current_price_tag = price_container.find('div', class_='prod-aslowas')
            current_price = current_price_tag.text if current_price_tag else 'Unknown'
        else:
            former_price = current_price = "Unknown"
    else:
        brand = name = former_price = current_price = "Unknown"

    data = {
        "Product_Name": name,
        "Brand": brand,
        "Former_Price": former_price,
        "Current_Price": current_price
    }
    print(data)
    products.append(data)

{'Product_Name': 'RB5154 Clubmaster', 'Brand': 'Ray-Ban', 'Former_Price': '$222', 'Current_Price': '$155.40'}
{'Product_Name': 'Airdrop', 'Brand': 'Oakley', 'Former_Price': '$227', 'Current_Price': '$158.90'}
{'Product_Name': 'Unknown', 'Brand': 'Unknown', 'Former_Price': 'Unknown', 'Current_Price': 'Unknown'}
{'Product_Name': 'RB7047', 'Brand': 'Ray-Ban', 'Former_Price': '$176', 'Current_Price': '$123.20'}
{'Product_Name': 'RB3547V Oval', 'Brand': 'Ray-Ban', 'Former_Price': '$210', 'Current_Price': '$147'}
{'Product_Name': 'Socket 5.5', 'Brand': 'Oakley', 'Former_Price': '$227', 'Current_Price': '$158.90'}
{'Product_Name': 'RB5387', 'Brand': 'Ray-Ban', 'Former_Price': '$176', 'Current_Price': '$123.20'}
{'Product_Name': 'Overhead', 'Brand': 'Oakley', 'Former_Price': '$172', 'Current_Price': '$120.40'}
{'Product_Name': 'Neoastra', 'Brand': 'Oakley', 'Former_Price': '$208', 'Current_Price': '$145.60'}
{'Product_Name': 'Thinboard', 'Brand': 'Oakley', 'Former_Price': '$172', 'Current_Pric

In [12]:
# Cell 9
# Step 3 - Data Storage: store the extracted data in CSV and JSON formats
import csv
import json
# Save to CSV file
column_name = products[0].keys() # get the column names
with open('glassesdotcom.csv', mode='w', newline='', encoding='utf-8') as csv_file: # open up the file with context manager
    dict_writer = csv.DictWriter(csv_file, fieldnames=column_name)
    dict_writer.writeheader()
    dict_writer.writerows(products)
print(f"Saved {len(products)} records to CSV in the extracted data folder.")

# Save to JSON file
with open("glassesdotcom.json", mode='w') as json_file:
    json.dump(products, json_file, indent=4)
print(f"Saved {len(products)} records to JSON in the extracted data folder.")

# close the browser
driver.quit()
print("End of Web Extraction")

Saved 26 records to CSV in the extracted data folder.
Saved 26 records to JSON in the extracted data folder.
End of Web Extraction
